In [4]:


import pandas as pd
import numpy as np


#  LOAD CLEANED DATA

df = pd.read_csv("../data/cleaned_data.csv")

print("✅ Data Loaded")
print("Shape:", df.shape)


# COLUMN DETECTION

def find_col(keywords):
    for col in df.columns:
        for k in keywords:
            if k in col:
                return col
    return None

revenue_col = find_col(['revenue','sales','amount'])
cost_col = find_col(['cost','purchase','price'])
inventory_col = find_col(['inventory','stock'])
vendor_col = find_col(['vendor','supplier'])

print("\nDetected Columns:")
print("Revenue:", revenue_col)
print("Cost:", cost_col)
print("Inventory:", inventory_col)
print("Vendor:", vendor_col)


# KPI DASHBOARD

print("\n" + "="*50)
print("KPI DASHBOARD")
print("="*50)

if revenue_col:
    print("Total Revenue:", df[revenue_col].sum())

if 'profit' in df.columns:
    print("Total Profit:", df['profit'].sum())
    print("Average Profit:", df['profit'].mean())

if 'profit_margin' in df.columns:
    print("Average Profit Margin:", df['profit_margin'].mean())

if 'inventory_turnover' in df.columns:
    print("Avg Inventory Turnover:", df['inventory_turnover'].mean())


# VENDOR PERFORMANCE SCORING

if 'profit_margin' in df.columns and 'inventory_turnover' in df.columns:

    df['vendor_score'] = (
        df['profit_margin'] * 0.5 +
        df['inventory_turnover'] * 0.3 +
        (df['profit'] / (df['profit'].max()+1)) * 0.2
    )

    print("\nVendor scoring created ✅")


#  TOP & LOW VENDORS

if vendor_col and 'vendor_score' in df.columns:

    vendor_rank = df.groupby(vendor_col)['vendor_score'].mean().sort_values(ascending=False)

    print("\nTop 10 Vendors:")
    print(vendor_rank.head(10))

    print("\nBottom 10 Vendors:")
    print(vendor_rank.tail(10))


# PROFIT SEGMENTATION

if 'profit' in df.columns:

    df['profit_segment'] = pd.qcut(
        df['profit'],
        q=4,
        labels=['Low','Mid-Low','Mid-High','High']
    )

    print("\nProfit Segments:")
    print(df['profit_segment'].value_counts())


#  CORRELATION ANALYSIS

print("\nCorrelation with Profit:")

if 'profit' in df.columns:
    corr = df.corr(numeric_only=True)
    print(corr['profit'].sort_values(ascending=False))

# ANOMALY DETECTION

if 'profit' in df.columns:

    z = (df['profit'] - df['profit'].mean()) / df['profit'].std()
    anomalies = df[np.abs(z) > 3]

    print("\nAnomalies Detected:", anomalies.shape[0])

# 10. INVENTORY RISK ANALYSIS

if 'inventory_turnover' in df.columns:

    risk = df[df['inventory_turnover'] < df['inventory_turnover'].quantile(0.25)]

    print("\nLow Inventory Turnover Count:", risk.shape[0])

if cost_col and revenue_col:

    df['cost_ratio'] = df[cost_col] / (df[revenue_col] + 1)

    print("\nCost Ratio Stats:")
    print(df['cost_ratio'].describe())

# VENDOR SUMMARY TABLE (SAFE FIX)

if vendor_col is None:
    print("\n⚠️ Vendor column not found — skipping vendor analysis")

else:
    agg_dict = {}

    if revenue_col:
        agg_dict[revenue_col] = 'sum'
    if 'profit' in df.columns:
        agg_dict['profit'] = 'sum'
    if 'profit_margin' in df.columns:
        agg_dict['profit_margin'] = 'mean'
    if 'inventory_turnover' in df.columns:
        agg_dict['inventory_turnover'] = 'mean'

  
    if len(agg_dict) == 0:
        print("\n⚠️ No valid columns for aggregation — skipping vendor summary")
    else:
        vendor_summary = df.groupby(vendor_col).agg(agg_dict)

        print("\nVendor Summary:")
        print(vendor_summary.head())

#  BUSINESS INSIGHTS

print("\n" + "="*50)
print("BUSINESS INSIGHTS")
print("="*50)

print("• Focus on high-performing vendors to maximize profitability.")
print("• Reduce dependency on low-margin vendors.")
print("• Improve inventory turnover to reduce holding costs.")
print("• Optimize cost structure to increase margins.")


# 14. SAVE OUTPUT

df.to_csv("../data/final_analysis.csv", index=False)



✅ Data Loaded
Shape: (100, 39)

Detected Columns:
Revenue: None
Cost: None
Inventory: None
Vendor: vendorid

KPI DASHBOARD

Correlation with Profit:

⚠️ No valid columns for aggregation — skipping vendor summary

BUSINESS INSIGHTS
• Focus on high-performing vendors to maximize profitability.
• Reduce dependency on low-margin vendors.
• Improve inventory turnover to reduce holding costs.
• Optimize cost structure to increase margins.
